# BIOSTAT707 Checkpoint 3

Name: Xiaoning Li
NetID: xl516

## AI-use statement

I used ChatGPT and Codex to help write Python code and organize the notebook. I reviewed the results and checked selected original measurements. 

I realized AI made coding faster, but its code and interpretations needed checking and sometimes included unnecessary detail.

## 3.1 Applying the model to set-b and set-c

I load the ridge model and preprocessing saved in Checkpoint 2. Neither is refitted or tuned on set-b or set-c.

Each row represents one intensive care unit (ICU) stay, summarized over hours 0–48. The saved pipeline fixes the imputation values, scaling, category encoding, and regression coefficients learned from set-a.


### Load packages and the saved model


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

SEED = 707

ROOT = Path.cwd()
assert (ROOT / "checkpoint3.ipynb").is_file(), "Run from the repository root."
DATA = ROOT / "data"
OUT = ROOT / "output"
SOURCE = DATA / "physionet.org/files/challenge-2012/1.0.0"
OUT.mkdir(exist_ok=True)

final_model = joblib.load(OUT / "cp2_final_model.joblib")

predictor_cols = pd.read_csv(
    OUT / "cp2_predictor_columns.csv"
)["Predictor"].tolist()

assert predictor_cols == final_model.feature_names_in_.tolist()

### Read records and outcomes

I check record IDs, the 0–48-hour window, and complete binary outcomes.


In [ ]:
def load_records(folder):
    files = sorted(folder.glob("*.txt"))
    assert len(files) == 4000, f"Expected 4000 files, found {len(files)}."

    tables = []

    for file in files:
        df = pd.read_csv(file)
        record_id = int(file.stem)

        recorded_id = df.loc[df["Parameter"] == "RecordID", "Value"]
        assert len(recorded_id) == 1
        assert recorded_id.iloc[0] == record_id

        df = df.loc[df["Parameter"] != "RecordID"].copy()
        df.insert(0, "RecordID", record_id)

        time = df["Time"].str.split(":", expand=True).astype(int)
        df["time_minutes"] = time[0] * 60 + time[1]
        tables.append(df)

    long = pd.concat(tables, ignore_index=True)

    assert long["RecordID"].nunique() == 4000
    assert long["time_minutes"].between(0, 48 * 60).all()

    return long


long_b = load_records(SOURCE / "set-b")
long_c = load_records(SOURCE / "set-c")

outcomes_b = pd.read_csv(SOURCE / "Outcomes-b.txt")
outcomes_c = pd.read_csv(SOURCE / "Outcomes-c.txt")

cohort_rows = []

for name, long, outcomes in [
    ("set-b", long_b, outcomes_b),
    ("set-c", long_c, outcomes_c)
]:
    assert outcomes["RecordID"].is_unique
    assert set(outcomes["RecordID"]) == set(long["RecordID"])
    assert outcomes["In-hospital_death"].isin([0, 1]).all()

    cohort_rows.append({
        "Set": name,
        "Stays": len(outcomes),
        "Deaths": int(outcomes["In-hospital_death"].sum()),
        "Death percentage": 100 * outcomes["In-hospital_death"].mean()
    })

cohort_summary = pd.DataFrame(cohort_rows)

display(
    cohort_summary.style
    .format({"Death percentage": "{:.2f}%"})
    .hide(axis="index")
)

Set-b has 4,000 stays and 568 deaths (14.20%). Set-c has 4,000 stays and 585 deaths (14.62%).

### Apply the Checkpoint 2 cleaning and feature definitions

I keep the same cleaning limits, summaries, and predictor order as Checkpoint 2. Missing measurements are handled by the saved set-a pipeline.


In [ ]:
def build_modeling_table(long, outcomes):
    long = long.copy()
    long["Value_clean"] = long["Value"].replace(-1, np.nan)

    bounds = {
        "Height": (50, 250),
        "Temp": (25, 45),
        "pH": (6, 8)
    }

    for variable, (lower, upper) in bounds.items():
        invalid = (
            long["Parameter"].eq(variable)
            & ~long["Value_clean"].between(lower, upper)
        )
        long.loc[invalid, "Value_clean"] = np.nan

    invalid_weight = (
        long["Parameter"].eq("Weight")
        & long["Value_clean"].eq(0)
    )
    long.loc[invalid_weight, "Value_clean"] = np.nan

    record_ids = pd.Index(
        sorted(long["RecordID"].unique()),
        name="RecordID"
    )
    static_vars = ["Age", "Gender", "Height", "ICUType"]

    static_rows = long.loc[long["Parameter"].isin(static_vars)]
    assert not static_rows.duplicated(["RecordID", "Parameter"]).any()

    static = static_rows.pivot(
        index="RecordID",
        columns="Parameter",
        values="Value_clean"
    ).reindex(record_ids)

    series = long.loc[
        ~long["Parameter"].isin(static_vars + ["MechVent"])
    ].sort_values(["RecordID", "time_minutes"], kind="stable")

    summary = (
        series.groupby(["RecordID", "Parameter"])["Value_clean"]
        .agg(["count", "last", "min", "max", "mean"])
        .unstack("Parameter")
    )

    summary.columns = [
        f"{variable}_{stat}" for stat, variable in summary.columns
    ]

    wide = static.join(summary).reindex(record_ids)

    vent = (
        long.loc[long["Parameter"].eq("MechVent")]
        .groupby("RecordID")["Value_clean"]
        .agg(["count", "max"])
        .rename(columns={
            "count": "MechVent_count",
            "max": "MechVent_ever_recorded"
        })
    )
    wide = wide.join(vent)

    # Keep the set-a feature definitions, even if a variable is absent.
    missing_cols = [
        col for col in predictor_cols
        if col.endswith("_never_measured")
    ]
    value_cols = [
        col for col in predictor_cols if col not in missing_cols
    ]

    for col in value_cols:
        if col not in wide.columns:
            wide[col] = np.nan

    count_cols = [col for col in value_cols if col.endswith("_count")]
    wide[count_cols] = wide[count_cols].fillna(0).astype(int)

    for col in missing_cols:
        variable = col.removesuffix("_never_measured")
        wide[col] = wide[f"{variable}_count"].eq(0).astype(int)

    wide = wide.reset_index().merge(
        outcomes,
        on="RecordID",
        how="left",
        validate="one_to_one"
    )
    wide.columns.name = None

    assert len(wide) == 4000
    assert wide["RecordID"].is_unique
    assert wide["In-hospital_death"].isin([0, 1]).all()

    return wide


wide_b = build_modeling_table(long_b, outcomes_b)
wide_c = build_modeling_table(long_c, outcomes_c)

### Predict mortality using the saved pipeline

Sex and ICU type use the same category labels as Checkpoint 2. I use the saved pipeline without fitting new preprocessing.


In [ ]:
def prepare_predictors(wide):
    X = wide[predictor_cols].copy()

    for col, allowed in {
        "Gender": [0, 1],
        "ICUType": [1, 2, 3, 4]
    }.items():
        assert X[col].dropna().isin(allowed).all()
        X[col] = X[col].replace({
            value: str(value) for value in allowed
        })

    assert X.columns.tolist() == final_model.feature_names_in_.tolist()
    return X


X_b = prepare_predictors(wide_b)
X_c = prepare_predictors(wide_c)

y_b = wide_b["In-hospital_death"].astype(int)
y_c = wide_c["In-hospital_death"].astype(int)

pred_b = final_model.predict_proba(X_b)[:, 1]
pred_c = final_model.predict_proba(X_c)[:, 1]

for predictions in [pred_b, pred_c]:
    assert np.isfinite(predictions).all()
    assert ((predictions >= 0) & (predictions <= 1)).all()

print("Set-b predictor shape:", X_b.shape)
print("Set-c predictor shape:", X_c.shape)

# Reconstruct the set-a reference from raw data; no CP2-generated table is required.
training_outcomes = pd.read_csv(DATA / "Outcomes-a.txt")
long_a = load_records(DATA / "set-a")
training_table = build_modeling_table(long_a, training_outcomes)
X_a = prepare_predictors(training_table)
assert X_a.columns.tolist() == final_model.feature_names_in_.tolist()
for evaluation_table in [wide_b, wide_c]:
    assert set(training_table["RecordID"]).isdisjoint(evaluation_table["RecordID"])
assert set(wide_b["RecordID"]).isdisjoint(wide_c["RecordID"])
print("Set-a reference rebuilt from raw data; predictor order matches the saved model and all three sets have disjoint stay IDs.")


## 3.2 Discrimination

I report area under the receiver operating characteristic curve (AUROC), with 95% confidence intervals (CIs) from 2,000 bootstrap samples of stays. Simplified Acute Physiology Score I (SAPS-I) and Sequential Organ Failure Assessment (SOFA) are evaluated as raw scores, excluding missing values and -1 codes. Their evaluation counts are shown in the table.

### Calculate AUROC and bootstrap confidence intervals


In [ ]:
from sklearn.metrics import roc_auc_score

def bootstrap_auc(y, scores, seed=SEED):
    y = np.asarray(y)
    scores = np.asarray(scores)

    observed = np.isfinite(scores)
    y = y[observed]
    scores = scores[observed]

    rng = np.random.default_rng(seed)
    bootstrap_scores = []

    for i in range(2000):
        sample = rng.integers(0, len(y), size=len(y))

        if np.unique(y[sample]).size < 2:
            continue

        bootstrap_scores.append(
            roc_auc_score(y[sample], scores[sample])
        )

    lower, upper = np.percentile(bootstrap_scores, [2.5, 97.5])

    return {
        "Stays": len(y),
        "Deaths": int(y.sum()),
        "AUROC": roc_auc_score(y, scores),
        "Lower 95% CI": lower,
        "Upper 95% CI": upper
    }


auc_rows = []

for name, wide, y, predictions in [
    ("set-b", wide_b, y_b, pred_b),
    ("set-c", wide_c, y_c, pred_c)
]:
    scores = {
        "Ridge": predictions,
        "SAPS-I": wide["SAPS-I"].replace(-1, np.nan),
        "SOFA": wide["SOFA"].replace(-1, np.nan)
    }

    for method, values in scores.items():
        result = bootstrap_auc(y, values)
        result["Set"] = name
        result["Method"] = method
        auc_rows.append(result)

auc_results = pd.DataFrame(auc_rows)[[
    "Set", "Method", "Stays", "Deaths",
    "AUROC", "Lower 95% CI", "Upper 95% CI"
]]

display(
    auc_results.style.format({
        "AUROC": "{:.3f}",
        "Lower 95% CI": "{:.3f}",
        "Upper 95% CI": "{:.3f}"
    }).hide(axis="index")
)

### Compare with Checkpoint 2 cross-validation

The table compares the set-a cross-validation (CV) mean and standard deviation (SD) with external AUROCs and bootstrap CIs. Fold SD and bootstrap CIs describe different sources of variation.


In [ ]:
cp2_comparison = pd.read_csv(
    OUT / "cp2_model_comparison.csv",
    index_col=0
)

comparison_rows = [{
    "Evaluation": "set-a CV",
    "AUROC": cp2_comparison.loc["Ridge", "Mean"],
    "CV SD": cp2_comparison.loc["Ridge", "SD"],
    "Lower 95% CI": np.nan,
    "Upper 95% CI": np.nan
}]

for name in ["set-b", "set-c"]:
    row = auc_results.loc[
        auc_results["Set"].eq(name)
        & auc_results["Method"].eq("Ridge")
    ].iloc[0]

    comparison_rows.append({
        "Evaluation": name,
        "AUROC": row["AUROC"],
        "CV SD": np.nan,
        "Lower 95% CI": row["Lower 95% CI"],
        "Upper 95% CI": row["Upper 95% CI"]
    })

external_comparison = pd.DataFrame(comparison_rows)

display(
    external_comparison.style.format({
        "AUROC": "{:.3f}",
        "CV SD": "{:.3f}",
        "Lower 95% CI": "{:.3f}",
        "Upper 95% CI": "{:.3f}"
    }, na_rep="—").hide(axis="index")
)

### Plot the precision-recall curve on set-b

The dashed line is the death-rate baseline. Precision is positive predictive value (PPV), and recall is sensitivity.


In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_b, pred_b)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(recall, precision, label="Ridge")
ax.axhline(
    y_b.mean(),
    color="gray",
    linestyle="--",
    label=f"Death rate = {y_b.mean():.1%}"
)

ax.set_xlabel("Recall (sensitivity)")
ax.set_ylabel("Precision (positive predictive value)")
ax.set_title("Precision-recall curve on set-b")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()

fig.tight_layout()
fig.savefig(
    OUT / "cp3_precision_recall.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

Ridge AUROC was 0.862 on set-b and 0.866 on set-c, above the set-a CV mean of 0.842. CV therefore did not appear optimistic on these sets. Benchmark scores were missing for some stays, so their AUROCs use smaller samples.

The precision-recall curve is above the 14.2% baseline across most of its range. Precision falls as recall rises: finding more deaths also flags more survivors. This curve is useful because deaths are a minority.


### Assess the set-b and set-c difference

I independently bootstrap each set to estimate a CI for set-c AUROC minus set-b AUROC.


In [ ]:
assert set(wide_b["RecordID"]).isdisjoint(wide_c["RecordID"])

rng = np.random.default_rng(SEED)
yb = y_b.to_numpy()
yc = y_c.to_numpy()
auc_differences = []

for i in range(2000):
    sample_b = rng.integers(0, len(yb), size=len(yb))
    sample_c = rng.integers(0, len(yc), size=len(yc))

    if (
        np.unique(yb[sample_b]).size < 2
        or np.unique(yc[sample_c]).size < 2
    ):
        continue

    auc_b = roc_auc_score(yb[sample_b], pred_b[sample_b])
    auc_c = roc_auc_score(yc[sample_c], pred_c[sample_c])
    auc_differences.append(auc_c - auc_b)

lower, upper = np.percentile(auc_differences, [2.5, 97.5])

difference_table = pd.DataFrame({
    "Comparison": ["set-c minus set-b"],
    "AUROC difference": [
        roc_auc_score(y_c, pred_c) - roc_auc_score(y_b, pred_b)
    ],
    "Lower 95% CI": [lower],
    "Upper 95% CI": [upper]
})

display(
    difference_table.style.format({
        "AUROC difference": "{:.3f}",
        "Lower 95% CI": "{:.3f}",
        "Upper 95% CI": "{:.3f}"
    }).hide(axis="index")
)

The AUROC difference was 0.004 (95% CI −0.016 to 0.026). The interval includes zero, so the difference is compatible with sampling variation.


## 3.3 Calibration

Ideal calibration has intercept 0 and slope 1. I estimate the intercept with predicted log odds as an offset, and the slope with a freely estimated intercept. The scaled Brier score compares squared prediction error with predicting each set's death rate for everyone.

### Calculate calibration metrics


In [ ]:
import statsmodels.api as sm

calibration_rows = []

for name, y, predictions in [
    ("set-b", y_b, pred_b),
    ("set-c", y_c, pred_c)
]:
    p = np.clip(predictions, 1e-6, 1 - 1e-6)
    logit_p = np.log(p / (1 - p))

    # Estimate the intercept with the predicted log odds as an offset.
    intercept_fit = sm.GLM(
        np.asarray(y),
        np.ones((len(y), 1)),
        offset=logit_p,
        family=sm.families.Binomial()
    ).fit()

    slope_fit = sm.GLM(
        np.asarray(y),
        sm.add_constant(logit_p),
        family=sm.families.Binomial()
    ).fit()

    brier = np.mean((np.asarray(y) - predictions) ** 2)
    baseline_brier = y.mean() * (1 - y.mean())

    calibration_rows.append({
        "Set": name,
        "Calibration intercept": intercept_fit.params[0],
        "Calibration slope": slope_fit.params[1],
        "Brier score": brier,
        "Scaled Brier score": 1 - brier / baseline_brier
    })

calibration_results = pd.DataFrame(calibration_rows)

display(
    calibration_results.style.format({
        "Calibration intercept": "{:.3f}",
        "Calibration slope": "{:.3f}",
        "Brier score": "{:.3f}",
        "Scaled Brier score": "{:.3f}"
    }).hide(axis="index")
)

### Plot flexible calibration curves

Locally weighted scatterplot smoothing (LOWESS) uses fraction 0.4 and 500 bootstrap samples for approximate 95% pointwise bands. Histograms show where predictions are concentrated. Curves and bands are clipped to 0–1 for display.


In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 7),
    sharex="col",
    gridspec_kw={"height_ratios": [3, 1]}
)

for column, (name, y, predictions) in enumerate([
    ("set-b", y_b, pred_b),
    ("set-c", y_c, pred_c)
]):
    outcomes = np.asarray(y)
    grid = np.linspace(predictions.min(), predictions.max(), 100)

    curve = lowess(
        outcomes,
        predictions,
        frac=0.4,
        it=0,
        xvals=grid
    )

    rng = np.random.default_rng(SEED)
    bootstrap_curves = []

    for i in range(500):
        sample = rng.integers(
            0, len(outcomes), size=len(outcomes)
        )

        bootstrap_curve = lowess(
            outcomes[sample],
            predictions[sample],
            frac=0.4,
            it=0,
            xvals=grid
        )
        bootstrap_curves.append(bootstrap_curve)

    lower, upper = np.percentile(
        bootstrap_curves, [2.5, 97.5], axis=0
    )

    ax = axes[0, column]
    ax.plot(grid, np.clip(curve, 0, 1), label="LOWESS")
    ax.fill_between(
        grid,
        np.clip(lower, 0, 1),
        np.clip(upper, 0, 1),
        alpha=0.2,
        label="95% confidence band"
    )
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set_title(name)
    ax.set_ylabel("Observed death rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()

    axes[1, column].hist(
        predictions,
        bins=np.linspace(0, 1, 21),
        edgecolor="white"
    )
    axes[1, column].set_xlabel("Predicted risk")
    axes[1, column].set_ylabel("Stays")

fig.tight_layout()
fig.savefig(
    OUT / "cp3_calibration_curves.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

Intercepts were near zero (−0.023 and 0.019), suggesting little average bias. Slopes were slightly above 1 (1.094 and 1.042), suggesting predictions were slightly less extreme than ideal.

Curves suggest underprediction around risks of 0.2–0.4 and overprediction at high risks. Few stays had high predicted risks, so the curve tails are uncertain. Brier scores were 0.088 and 0.092; scaled scores of 0.278 and 0.265 indicate about 28% and 27% less squared error than the constant death-rate predictions.


## 3.4 Clinical usefulness

I propose a 20% mortality-risk threshold for additional clinical review after 48 hours. It was specified before examining decision curves. I compare thresholds of 5%–40%.

SOFA is converted to probability using logistic regression fitted only on set-a. All decision curves use the same set-b stays with valid SOFA scores.

### Convert SOFA to probability using set-a

The 20% threshold is an illustrative trade-off: one true-positive review balances four false-positive reviews under the net-benefit weighting. Additional review is a plausible action at 48 hours, but the threshold needs prospective workload and benefit assessment before implementation. Mapping SOFA to probability on set-a permits comparison at the same risk thresholds without fitting to test outcomes.


In [ ]:
outcomes_a = pd.read_csv(DATA / "Outcomes-a.txt")

sofa_a = outcomes_a["SOFA"].replace(-1, np.nan)
observed_a = sofa_a.notna()

sofa_model = sm.GLM(
    outcomes_a.loc[observed_a, "In-hospital_death"].to_numpy(),
    sm.add_constant(sofa_a[observed_a].to_numpy()),
    family=sm.families.Binomial()
).fit()

sofa_b = wide_b["SOFA"].replace(-1, np.nan)
observed_b = sofa_b.notna().to_numpy()

sofa_risk_b = sofa_model.predict(
    sm.add_constant(sofa_b[observed_b].to_numpy())
)

dca_y = y_b.to_numpy()[observed_b]
dca_ridge = pred_b[observed_b]

### Plot decision curves on set-b

Net benefit weights false positives by threshold odds. “Treat all” and “treat none” mean flagging everyone or no one for review.


In [ ]:
def net_benefit(y, predictions, threshold):
    flagged = predictions >= threshold

    true_positives = np.sum(flagged & (y == 1))
    false_positives = np.sum(flagged & (y == 0))

    return (
        true_positives / len(y)
        - false_positives / len(y) * threshold / (1 - threshold)
    )


thresholds = np.linspace(0.05, 0.40, 71)
dca_rows = []

for threshold in thresholds:
    prevalence = dca_y.mean()

    dca_rows.append({
        "Threshold": threshold,
        "Ridge": net_benefit(dca_y, dca_ridge, threshold),
        "SOFA": net_benefit(dca_y, sofa_risk_b, threshold),
        "Treat all": (
            prevalence
            - (1 - prevalence) * threshold / (1 - threshold)
        ),
        "Treat none": 0
    })

dca_results = pd.DataFrame(dca_rows)

fig, ax = plt.subplots(figsize=(7, 5))

for method in ["Ridge", "SOFA", "Treat all", "Treat none"]:
    ax.plot(
        dca_results["Threshold"],
        dca_results[method],
        label=method
    )

ax.set_xlabel("Risk threshold")
ax.set_ylabel("Net benefit")
ax.set_title(f"Decision curves on set-b (n = {len(dca_y):,})")
ax.legend()

fig.tight_layout()
fig.savefig(
    OUT / "cp3_decision_curves.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

Among 3,863 stays with valid SOFA scores, ridge had higher net benefit than SOFA, flagging everyone, and flagging no one across 5%–40%. This supports potential use for review under the assumed trade-off, but does not prove improved patient outcomes.


### Evaluate the 20% threshold

Metrics use all set-b stays. The Challenge 2012 event 1 score is min(sensitivity, PPV).


In [ ]:
threshold = 0.20
outcomes = y_b.to_numpy()
flagged = pred_b >= threshold

true_positives = np.sum(flagged & (outcomes == 1))

sensitivity = true_positives / np.sum(outcomes == 1)
ppv = true_positives / flagged.sum() if flagged.sum() > 0 else np.nan

threshold_results = pd.DataFrame({
    "Threshold": [threshold],
    "Sensitivity": [sensitivity],
    "PPV": [ppv],
    "Flagged per 100": [100 * flagged.mean()],
    "Event 1 score": [min(sensitivity, ppv)]
})

display(
    threshold_results.style.format({
        "Threshold": "{:.0%}",
        "Sensitivity": "{:.3f}",
        "PPV": "{:.3f}",
        "Flagged per 100": "{:.1f}",
        "Event 1 score": "{:.3f}"
    }).hide(axis="index")
)

At 20%, the model flagged 22.9 per 100 patients, with sensitivity 0.701, PPV 0.435, and event 1 score 0.435. It identified about 70% of deaths, but more than half of flagged patients survived.


## 3.5 Subgroups

I compare age groups (under 65 and 65 or older), sex, and ICU type on set-b. Calibration intercepts use predicted log odds as an offset.

### Calculate subgroup performance


In [ ]:
subgroups = wide_b[
    ["Age", "Gender", "ICUType", "In-hospital_death"]
].copy()

subgroups["Predicted risk"] = pred_b

subgroups["Age group"] = pd.cut(
    subgroups["Age"],
    bins=[-np.inf, 65, np.inf],
    labels=["Under 65", "65 or older"],
    right=False
).astype("object").fillna("Unknown")

subgroups["Sex"] = (
    subgroups["Gender"]
    .map({0: "Female", 1: "Male"})
    .fillna("Unknown")
)

subgroups["ICU type"] = (
    subgroups["ICUType"]
    .map({
        1: "Coronary care",
        2: "Cardiac surgery recovery",
        3: "Medical",
        4: "Surgical"
    })
    .fillna("Unknown")
)

subgroup_rows = []

for category in ["Age group", "Sex", "ICU type"]:
    for group, patients in subgroups.groupby(category, sort=False):
        outcomes = patients["In-hospital_death"].to_numpy()
        predictions = patients["Predicted risk"].to_numpy()

        auc = np.nan
        intercept = np.nan

        # Both outcomes are needed for these estimates.
        if np.unique(outcomes).size == 2:
            auc = roc_auc_score(outcomes, predictions)

            p = np.clip(predictions, 1e-6, 1 - 1e-6)
            logit_p = np.log(p / (1 - p))

            fit = sm.GLM(
                outcomes,
                np.ones((len(outcomes), 1)),
                offset=logit_p,
                family=sm.families.Binomial()
            ).fit()

            intercept = fit.params[0]

        subgroup_rows.append({
            "Category": category,
            "Group": group,
            "Stays": len(patients),
            "Deaths": int(outcomes.sum()),
            "AUROC": auc,
            "Calibration intercept": intercept
        })

subgroup_results = pd.DataFrame(subgroup_rows)

display(
    subgroup_results.style.format({
        "AUROC": "{:.3f}",
        "Calibration intercept": "{:.3f}"
    }, na_rep="Not estimable").hide(axis="index")
)

AUROC was lower at age 65 or older (0.835 versus 0.886); male and female values were similar (0.863 and 0.861). Their calibration intercepts were near zero.

ICU AUROCs ranged from 0.828 to 0.872. Cardiac surgery recovery showed the largest average overprediction (intercept −0.438), but had only 34 deaths. The five stays with unknown sex had no deaths, so estimates were unavailable. Age and cardiac surgery differences deserve attention, but case mix, small groups, and multiple comparisons limit firm conclusions.


## 3.6 Interpretation

Coefficients are expressed per one standard deviation of each transformed predictor on set-a.

### Plot the largest standardized coefficients

Clinical variable labels used in tables and plots include heart rate (HR), Glasgow Coma Scale (GCS), mean arterial pressure (MAP), non-invasive mean arterial pressure (NIMAP), blood urea nitrogen (BUN), white blood cell count (WBC), hematocrit (HCT), bicarbonate (HCO3), alkaline phosphatase (ALP), alanine aminotransferase (ALT), aspartate aminotransferase (AST), arterial oxygen and carbon dioxide pressures (PaO2 and PaCO2), and fraction of inspired oxygen (FiO2). Suffixes describe the summary, such as mean, last valid value, minimum, maximum, or count.


In [ ]:
preprocessing = final_model.named_steps["preprocess"]
transformed_training = preprocessing.transform(prepare_predictors(training_table))
coefficients = pd.DataFrame({
    "Predictor": [name.split("__", 1)[-1] for name in preprocessing.get_feature_names_out()],
    "Standardized coefficient": final_model.named_steps["model"].coef_[0] * transformed_training.std(axis=0)
})
coefficients = coefficients.iloc[
    np.argsort(-coefficients["Standardized coefficient"].abs().to_numpy())
].reset_index(drop=True)

# Check the reconstructed reference scale against the committed CP2 coefficients.
saved_coefficients = pd.read_csv(OUT / "cp2_ridge_coefficients.csv").set_index("Predictor")
current_coefficients = coefficients.set_index("Predictor")
assert set(saved_coefficients.index) == set(current_coefficients.index)
np.testing.assert_allclose(
    current_coefficients.loc[saved_coefficients.index, "Standardized coefficient"],
    saved_coefficients["Standardized coefficient"], rtol=1e-8, atol=1e-10
)

top_coefficients = coefficients.head(20).sort_values(
    "Standardized coefficient"
)

fig, ax = plt.subplots(figsize=(8, 7))

ax.barh(
    top_coefficients["Predictor"],
    top_coefficients["Standardized coefficient"]
)

ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Log odds change per one standard deviation")
ax.set_title("Ridge: 20 largest standardized coefficients")

fig.tight_layout()
fig.savefig(
    OUT / "cp3_coefficients.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

Age had a large positive coefficient and last Glasgow Coma Scale (GCS) had the largest negative coefficient. Counts and missingness indicators also contributed.

These are associations. Missing ventilation records may reflect documentation or treatment decisions, while measurement counts may reflect monitoring intensity. Changing documentation would not necessarily change outcomes. Correlated summaries also make individual coefficients difficult to interpret.


## 3.7 Most confident mistakes

I examine five highest-risk survivors and five lowest-risk deaths. The table shows each stay's three largest contributions toward the incorrect outcome, relative to average transformed set-a predictors. Missingness is calculated across static and measurement-value input columns before imputation, excluding counts and never-measured indicators that are always observed. It is not the proportion of missing raw measurement rows.

### Identify confident mistakes and their main predictors


In [ ]:
patients = wide_b[
    ["RecordID", "In-hospital_death"]
].copy()
patients["Predicted risk"] = pred_b

high_risk_survivors = (
    patients.loc[patients["In-hospital_death"].eq(0)]
    .nlargest(5, "Predicted risk")
)

low_risk_deaths = (
    patients.loc[patients["In-hospital_death"].eq(1)]
    .nsmallest(5, "Predicted risk")
)

mistakes = pd.concat([
    high_risk_survivors.assign(Type="High-risk survivor"),
    low_risk_deaths.assign(Type="Low-risk death")
])

# Use set-a as the reference for coefficient contributions.
# Reuse the reference reconstructed in Section 3.1.
wide_a = training_table

preprocessing = final_model.named_steps["preprocess"]
transformed_a = preprocessing.transform(X_a)
transformed_b = preprocessing.transform(X_b)

reference = transformed_a.mean(axis=0)
beta = final_model.named_steps["model"].coef_[0]
contributions = (transformed_b - reference) * beta

feature_names = (
    pd.Series(preprocessing.get_feature_names_out())
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
    .to_numpy()
)

mistakes["Missing value predictors (%)"] = (
    X_b.loc[mistakes.index, [col for col in predictor_cols
        if not col.endswith(("_count", "_never_measured"))]].isna().mean(axis=1) * 100
)

display(
    mistakes.style.format({
        "Predicted risk": "{:.3f}",
        "Missing value predictors (%)": "{:.1f}"
    }).hide(axis="index")
)

driver_rows = []

for index, patient in mistakes.iterrows():
    values = contributions[index]

    if patient["Type"] == "High-risk survivor":
        selected = np.argsort(values)[-3:][::-1]
    else:
        selected = np.argsort(values)[:3]

    for position in selected:
        feature = feature_names[position]

        # Encoded categories are shown on their transformed 0/1 scale.
        if feature in X_b.columns:
            observed_value = X_b.loc[index, feature]
        else:
            observed_value = transformed_b[index, position]

        driver_rows.append({
            "RecordID": int(patient["RecordID"]),
            "Type": patient["Type"],
            "Predictor": feature,
            "Value": observed_value,
            "Log odds contribution": values[position]
        })

mistake_drivers = pd.DataFrame(driver_rows)

display(
    mistake_drivers.style.format({
        "Log odds contribution": "{:.3f}"
    }, na_rep="Missing").hide(axis="index")
)

reference_log_odds = final_model.named_steps["model"].intercept_[0] + reference @ beta
assert np.allclose(reference_log_odds + contributions.sum(axis=1), final_model.decision_function(X_b))


### Check unusual measurements

I inspect magnesium and weight for two low-risk deaths. Repeated identical values are shown once.


In [ ]:
for record_id, variable in [
    (142675, "Mg"),
    (152270, "Weight")
]:
    measurements = long_b.loc[
        long_b["RecordID"].eq(record_id)
        & long_b["Parameter"].eq(variable),
        ["RecordID", "Time", "Parameter", "Value"]
    ]

    display(
        measurements.drop_duplicates(["Parameter", "Value"])
        .style.hide(axis="index")
    )

Magnesium of 21 fell to 2.5 within 53 minutes, suggesting a possible recording or measurement problem. Weight of 295 was repeated throughout the window, which does not confirm accuracy. Neither value was changed.

| RecordID | Main prediction drivers | Possible explanation |
|---|---|---|
| 148973 | High bilirubin, low GCS, older age | Severe features raised risk, but recovery or treatment information may be missing. No specific data error was identified. |
| 144653 | Low GCS, high lactate, low pH | The model may miss treatment response or recovery; abnormal features do not make death certain. |
| 142993 | Low GCS, high bilirubin and blood urea nitrogen (BUN) | The summaries indicate high risk but do not explain survival. Model or information limits are possible. |
| 147603 | High white blood cell (WBC) count, older age, low pH | Possible overprediction or unrecorded recovery; no confirmed data error. |
| 149401 | High bilirubin summaries, low GCS | Correlated summaries raised risk. Recovery or model limitations may explain survival. |
| 142675 | High maximum magnesium and measurement counts lowered risk | A possible magnesium recording problem strongly affected the prediction, showing sensitivity to an extreme value. |
| 152270 | Young age, high weight, and mean arterial carbon dioxide pressure lowered risk | Weight needs verification. Model limitations or information outside the window may also matter. |
| 150189 | GCS of 15, temperature count, ventilation documentation | Reassuring and monitoring features lowered risk. Later deterioration or missing clinical information is possible. |
| 152476 | Temperature count, GCS of 15, ventilation documentation | Monitoring features lowered risk; the cause of death cannot be established from these data. |
| 143448 | GCS of 15, hematocrit count, ventilation documentation | The model may miss clinical information or later changes. Missing predictors may contribute. |

These are possible explanations, not confirmed causes. An unexpected outcome does not by itself prove a probability estimate was wrong.

The case tables identify which recorded predictors contributed most strongly to each extreme prediction. Their mixture of plausible severity markers, monitoring features, and possible recording errors suggests that both incomplete clinical information and model sensitivity can explain unexpected outcomes; none of these case reviews changes the fitted model.


## 3.8 Written conclusions

Ridge discriminated well on set-b and set-c (AUROC 0.862 and 0.866). Average calibration bias was small, with some local miscalibration. Decision curves favored ridge over the alternatives examined.

I suggest further evaluation for additional clinical review at 20%, rather than routine implementation yet. It flags about 23 per 100 patients and identifies 70% of deaths, but more than half of flagged patients survive.

Limitations include one data source, missing measurements, possible recording errors, and unavailable diagnoses, treatment responses, or later events. The 48-hour window prevents use at admission. Subgroup performance also varied.

Leakage could enter through post-window information, preprocessing fitted on evaluation data, or tuning after observing test performance. RecordID, length of stay, survival time, and the outcome are excluded from predictors; SAPS-I and SOFA are separate benchmarks. The saved set-a pipeline is applied without refitting on set-b or set-c, and SOFA probability conversion uses set-a only. The set-a reference is reconstructed from raw data without refitting the model. Predictor order is checked against the saved model, and the reconstructed reference scale is checked against the saved Checkpoint 2 standardized coefficients. Measurement counts and ventilation documentation remain predictive associations, not causal treatment effects.
